In [1]:
import redis
import json
import time
import os
import uuid
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

r = redis.Redis(
    host=os.getenv('REDIS_HOST', 'localhost'),
    port=int(os.getenv('REDIS_PORT', 6379)),
    decode_responses=True
)

groq_client = Groq(api_key=os.getenv('GROQ_API_KEY'))

r.ping()
print("Redis connected!")
print(f"Groq API key loaded: {'Yes' if os.getenv('GROQ_API_KEY') else 'NO - check your .env file!'}")

Redis connected!
Groq API key loaded: Yes


In [2]:
# ─────────────────────────────────────────────────
# The ConversationMemory class
# Handles all Redis operations for chat history
# ─────────────────────────────────────────────────

class ConversationMemory:
    def __init__(
        self,
        redis_client,
        max_messages: int = 20,
        session_ttl_seconds: int = 7200
    ):
        self.r = redis_client
        self.max_messages = max_messages
        self.session_ttl = session_ttl_seconds

    def _key(self, session_id: str) -> str:
        return f"conversation:{session_id}"

    def add_message(self, session_id: str, role: str, content: str):
        """Append a message to the session's conversation list."""
        key = self._key(session_id)
        message = json.dumps({
            "role": role,
            "content": content,
            "timestamp": time.time()
        })
        self.r.rpush(key, message)
        # Enforce sliding window — keep only the last N messages
        self.r.ltrim(key, -self.max_messages, -1)
        # Reset session TTL on every new message
        self.r.expire(key, self.session_ttl)

    def get_messages(self, session_id: str) -> list:
        """Retrieve all messages in the session, oldest first."""
        key = self._key(session_id)
        raw = self.r.lrange(key, 0, -1)
        return [json.loads(m) for m in raw]

    def get_message_count(self, session_id: str) -> int:
        return self.r.llen(self._key(session_id))

    def clear_session(self, session_id: str):
        """Delete the entire conversation history for a session."""
        self.r.delete(self._key(session_id))

    def get_session_ttl(self, session_id: str) -> int:
        """How many seconds until this session expires."""
        return self.r.ttl(self._key(session_id))


memory = ConversationMemory(r, max_messages=20, session_ttl_seconds=7200)

# ─────────────────────────────────────────────
# PRETTY OUTPUT
# ─────────────────────────────────────────────
print("\n" + "═" * 62)
print("\033[1;36m  💬  CONVERSATION MEMORY\033[0m")
print("\033[90m  Redis-backed sliding-window chat history\033[0m")
print("═" * 62)

print(f"\n  \033[1;33m▸ Max messages\033[0m     → \033[1;37m{memory.max_messages}\033[0m  \033[90m(sliding window)\033[0m")
print(f"  \033[1;33m▸ Session TTL\033[0m      → \033[1;37m{memory.session_ttl}\033[0m sec  \033[90m({memory.session_ttl // 3600}h)\033[0m")
print(f"  \033[1;33m▸ Key prefix\033[0m       → \033[90mconversation:<session_id>\033[0m")

print(f"\n  \033[1;32m✅ Available methods:\033[0m")
print(f"     \033[90m• add_message(session_id, role, content)\033[0m")
print(f"     \033[90m• get_messages(session_id)\033[0m")
print(f"     \033[90m• get_message_count(session_id)\033[0m")
print(f"     \033[90m• clear_session(session_id)\033[0m")
print(f"     \033[90m• get_session_ttl(session_id)\033[0m")

print("\n" + "═" * 62 + "\n")


══════════════════════════════════════════════════════════════
  💬  CONVERSATION MEMORY
  Redis-backed sliding-window chat history
══════════════════════════════════════════════════════════════

  ▸ Max messages     → 20  (sliding window)
  ▸ Session TTL      → 7200 sec  (2h)
  ▸ Key prefix       → conversation:<session_id>

  ✅ Available methods:
     • add_message(session_id, role, content)
     • get_messages(session_id)
     • get_message_count(session_id)
     • clear_session(session_id)
     • get_session_ttl(session_id)

══════════════════════════════════════════════════════════════



In [ ]:
# ─────────────────────────────────────────────────
# The chat function — ties memory + Groq together
# ─────────────────────────────────────────────────

def chat(
    session_id: str,
    user_message: str,
    system_prompt: str = "You are a helpful, friendly AI assistant. Keep responses concise."
) -> str:
    """Send a message and get a reply, with full conversation history from Redis."""

    # 1. Store the user's message in Redis
    memory.add_message(session_id, "user", user_message)

    # 2. Build the full message list: system prompt + all history
    messages = [{"role": "system", "content": system_prompt}]
    history = memory.get_messages(session_id)
    messages.extend([
        {"role": m["role"], "content": m["contengroqt"]}
        for m in history
    ])

    # 3. Call Groq with full history
    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-120b",
        messages=messages,
        max_tokens=400
    )

    reply = response.choices[0].message.content

    # 4. Store the assistant's reply in Redis
    memory.add_message(session_id, "assistant", reply)

    return reply


print("chat() function ready!")

chat() function ready!


In [4]:
# Start a new session
session_id = "demo_session_001"
memory.clear_session(session_id)  # Start fresh

def show_chat(session_id, user_msg):
    """Send a message and display it nicely."""
    print(f"\n[You]        {user_msg}")
    reply = chat(session_id, user_msg)
    print(f"[Assistant]  {reply}")
    print(f"(History: {memory.get_message_count(session_id)} messages in Redis)")

# Conversation Turn 1
show_chat(session_id, "Hi! My name is Mustafa.")


[You]        Hi! My name is Mustafa.
[Assistant]  Hi Mustafa! Nice to meet you. How can I help you today?
(History: 2 messages in Redis)


In [5]:
# Conversation Turn 2 — will the bot remember?
show_chat(session_id, "What is my name?")


[You]        What is my name?
[Assistant]  Your name is Mustafa.
(History: 4 messages in Redis)


In [6]:
# Conversation Turn 4 — building on context
show_chat(session_id, "Can you give me a quick summary of what we've talked about so far?")


[You]        Can you give me a quick summary of what we've talked about so far?
[Assistant]  Sure! Here’s a brief recap:

1. You introduced yourself as **Mustafa**.  
2. I greeted you and asked how I could help.  
3. You asked me to confirm your name, and I replied that your name is **Mustafa**.  

That’s where we’re at so far. Let me know what you’d like to discuss next!
(History: 6 messages in Redis)


In [7]:
# Inspect what's actually stored in Redis
print("=" * 55)
print("WHAT'S STORED IN REDIS FOR THIS SESSION")
print("=" * 55)

messages = memory.get_messages(session_id)
print(f"Total messages stored: {len(messages)}")
print(f"Session TTL: {memory.get_session_ttl(session_id)} seconds remaining")
print()

for i, msg in enumerate(messages):
    ts = time.strftime('%H:%M:%S', time.localtime(msg['timestamp']))
    role_label = f"[{msg['role'].upper():10}]"
    content_preview = msg['content'][:80] + "..." if len(msg['content']) > 80 else msg['content']
    print(f"  {i+1}. {ts} {role_label} {content_preview}")

WHAT'S STORED IN REDIS FOR THIS SESSION
Total messages stored: 6
Session TTL: 7084 seconds remaining

  1. 15:57:02 [USER      ] Hi! My name is Mustafa.
  2. 15:57:03 [ASSISTANT ] Hi Mustafa! Nice to meet you. How can I help you today?
  3. 15:58:07 [USER      ] What is my name?
  4. 15:58:08 [ASSISTANT ] Your name is Mustafa.
  5. 15:59:08 [USER      ] Can you give me a quick summary of what we've talked about so far?
  6. 15:59:09 [ASSISTANT ] Sure! Here’s a brief recap:

1. You introduced yourself as **Mustafa**.  
2. I g...


In [8]:
# Simulate two separate users chatting at the same time
# Their histories are completely separate in Redis

session_user1 = "user_session_mustafa"
session_user2 = "user_session_bob"

memory.clear_session(session_user1)
memory.clear_session(session_user2)

print("\n" + "═" * 70)
print("\033[1;36m  👤  MUSTAFA's CONVERSATION\033[0m")
print("═" * 70)

print(f"\n  \033[1;36m[MUSTAFA]\033[0m I'm Mustafa and I work as a data scientist.")
reply = chat(session_user1, "I'm Mustafa and I work as a data scientist.")
print(f"  \033[1;32m[BOT]    \033[0m {reply}")

# ─────────────────────────────────────────────
print("\n" + "═" * 70)
print("\033[1;35m  👤  BOB's CONVERSATION\033[0m \033[90m(separate session)\033[0m")
print("═" * 70)

print(f"\n  \033[1;35m[BOB]    \033[0m Hey! I'm Bob and I love cooking.")
reply = chat(session_user2, "Hey! I'm Bob and I love cooking.")
print(f"  \033[1;32m[BOT]    \033[0m {reply}")

# ─────────────────────────────────────────────
print("\n" + "═" * 70)
print("\033[1;33m  🔒  ISOLATION TEST — Mustafa asks about himself\033[0m")
print("\033[90m  Expected: knows his name and profession\033[0m")
print("═" * 70)

print(f"\n  \033[1;36m[MUSTAFA]\033[0m What's my name and what do I do?")
reply = chat(session_user1, "What's my name and what do I do?")
print(f"  \033[1;32m[BOT]    \033[0m {reply}")

# ─────────────────────────────────────────────
print("\n" + "═" * 70)
print("\033[1;33m  🔒  ISOLATION TEST — Bob asks about Alice\033[0m")
print("\033[90m  Expected: does NOT know about Alice\033[0m")
print("═" * 70)

print(f"\n  \033[1;35m[BOB]    \033[0m Do you know someone named Alice?")
reply = chat(session_user2, "Do you know someone named Alice?")
print(f"  \033[1;32m[BOT]    \033[0m {reply}")

# ─────────────────────────────────────────────
# REDIS SESSION KEYS
# ─────────────────────────────────────────────
print("\n" + "─" * 70)
print("\033[1;34m  🗄️   REDIS SESSION KEYS\033[0m")
print("─" * 70)

alice_count = memory.get_message_count(session_user1)
bob_count   = memory.get_message_count(session_user2)

print(f"\n  \033[1;36m▸ Mustafa\033[0m  → \033[90mconversation:{session_user1}\033[0m  "
      f"(\033[1;37m{alice_count}\033[0m msgs)")
print(f"  \033[1;35m▸ Bob\033[0m      → \033[90mconversation:{session_user2}\033[0m  "
      f"(\033[1;37m{bob_count}\033[0m msgs)")

print("\n" + "═" * 70 + "\n")


══════════════════════════════════════════════════════════════════════
  👤  MUSTAFA's CONVERSATION
══════════════════════════════════════════════════════════════════════

  [MUSTAFA] I'm Mustafa and I work as a data scientist.
  [BOT]     Nice to meet you, Mustafa! How's data science treating you these days? Let me know if there's anything I can help with.

══════════════════════════════════════════════════════════════════════
  👤  BOB's CONVERSATION (separate session)
══════════════════════════════════════════════════════════════════════

  [BOB]     Hey! I'm Bob and I love cooking.
  [BOT]     Hey Bob! 👋 That’s awesome—cooking is such a fun creative outlet. What’s your favorite dish to make, or is there something new you’re experimenting with?

══════════════════════════════════════════════════════════════════════
  🔒  ISOLATION TEST — Mustafa asks about himself
  Expected: knows his name and profession
══════════════════════════════════════════════════════════════════════

  [MUSTA

## Sliding Window

In [9]:
small_window_session = "demo_small_window"
small_memory = ConversationMemory(r, max_messages=6, session_ttl_seconds=300)
small_memory.clear_session(small_window_session)

turns = [
    "Turn 1: What is Python?",
    "Turn 2: What is JavaScript?",
    "Turn 3: What is Go?",
    "Turn 4: What is Rust?",
]

for msg in turns:
    small_memory.add_message(small_window_session, "user", msg)
    small_memory.add_message(small_window_session, "assistant", f"Response to: {msg}")

print("Messages added! GO CHECK REDIS DASHBOARD NOW")
print(f"Look for key: conversation:{small_window_session}")


Messages added! GO CHECK REDIS DASHBOARD NOW
Look for key: conversation:demo_small_window


In [10]:
# ─────────────────────────────────────────────
# SLIDING WINDOW — VERIFICATION
# ─────────────────────────────────────────────
msg_count = small_memory.get_message_count(small_window_session)

print("\n" + "═" * 70)
print("\033[1;36m  🪟  SLIDING WINDOW — VERIFICATION\033[0m")
print("═" * 70)

print(f"\n  \033[1;34m▸ Total messages in Redis\033[0m → \033[1;37m{msg_count}\033[0m")
print(f"  \033[90m  (max window was 6, so Turn 1 was pushed out)\033[0m")

# ─────────────────────────────────────────────
# OLDEST MESSAGE IN WINDOW
# ─────────────────────────────────────────────
oldest = json.loads(r.lrange(f"conversation:{small_window_session}", 0, 0)[0])["content"]

print(f"\n  \033[1;33m▸ Oldest message now\033[0m")
print(f"     \033[1;37m'{oldest}'\033[0m")

print("\n" + "═" * 70 + "\n")


══════════════════════════════════════════════════════════════════════
  🪟  SLIDING WINDOW — VERIFICATION
══════════════════════════════════════════════════════════════════════

  ▸ Total messages in Redis → 6
    (max window was 6, so Turn 1 was pushed out)

  ▸ Oldest message now
     'Turn 2: What is JavaScript?'

══════════════════════════════════════════════════════════════════════



In [11]:
# ─────────────────────────────────────────────
# FINAL WINDOW — LAST 6 MESSAGES
# ─────────────────────────────────────────────
messages = small_memory.get_messages(small_window_session)

print("\n" + "═" * 72)
print("\033[1;36m  🪟  FINAL WINDOW\033[0m")
print(f"\033[90m  Showing last {len(messages)} messages in the sliding window\033[0m")
print("═" * 72)

if not messages:
    print(f"\n  \033[1;33m⚠️  No messages in this session.\033[0m\n")
else:
    print()
    for idx, msg in enumerate(messages, 1):
        role = msg['role']
        content = msg['content']

        # Determine color and label based on role
        if role == "user":
            role_display = "\033[1;36m[USER]     \033[0m"
        elif role == "assistant":
            role_display = "\033[1;32m[ASSISTANT]\033[0m"
        elif role == "system":
            role_display = "\033[1;35m[SYSTEM]   \033[0m"
        else:
            role_display = f"\033[1;37m[{role.upper():<10}]\033[0m"

        # Message number in gray
        print(f"  \033[90m{idx:>2}.\033[0m {role_display} \033[90m{content}\033[0m")

print("\n" + "═" * 72 + "\n")


════════════════════════════════════════════════════════════════════════
  🪟  FINAL WINDOW
  Showing last 0 messages in the sliding window
════════════════════════════════════════════════════════════════════════

  ⚠️  No messages in this session.


════════════════════════════════════════════════════════════════════════



## Long Term Memory

In [14]:
class UserProfile:
    """Persistent user facts stored in Redis Hashes. No auto-expiry."""

    def __init__(self, redis_client):
        self.r = redis_client

    def _key(self, user_id: str) -> str:
        return f"profile:{user_id}"

    def set_fact(self, user_id: str, field: str, value: str):
        self.r.hset(self._key(user_id), field, value)

    def get_fact(self, user_id: str, field: str) -> str:
        return self.r.hget(self._key(user_id), field) or ""

    def get_all_facts(self, user_id: str) -> dict:
        return self.r.hgetall(self._key(user_id))

    def delete_fact(self, user_id: str, field: str):
        self.r.hdel(self._key(user_id), field)

    def clear(self, user_id: str):
        self.r.delete(self._key(user_id))


def chat_with_profile(
    session_id: str,
    user_id: str,
    user_message: str,
    profile: UserProfile
) -> str:
    """Chat that includes the user's persistent profile in the system prompt."""
    facts = profile.get_all_facts(user_id)

    if facts:
        facts_text = "\n".join([f"- {k}: {v}" for k, v in facts.items()])
        system_prompt = f"""You are a helpful AI assistant.

Known facts about this user:
{facts_text}

Use these facts naturally in your responses when relevant."""
    else:
        system_prompt = "You are a helpful AI assistant."

    return chat(session_id, user_message, system_prompt)


profile = UserProfile(r)

# Store some facts about the user
profile.set_fact("user_mustafa", "name", "Mustafa")
profile.set_fact("user_mustafa", "role", "Machine Learning Engineer")
profile.set_fact("user_mustafa", "experience", "3 years")
profile.set_fact("user_mustafa", "preferred_language", "Python")

# ─────────────────────────────────────────────
# PRETTY OUTPUT — STORED PROFILE
# ─────────────────────────────────────────────
facts = profile.get_all_facts("user_mustafa")

print("\n" + "═" * 62)
print("\033[1;36m  👤  USER PROFILE — STORED FACTS\033[0m")
print(f"\033[90m  Redis Hash: profile:user_mustafa\033[0m")
print("═" * 62)

if not facts:
    print(f"\n  \033[1;33m⚠️  No facts stored for this user.\033[0m\n")
else:
    print(f"\n  \033[1;33m{'FIELD':<25} {'VALUE'}\033[0m")
    print("  " + "─" * 56)

    for k, v in facts.items():
        print(f"  \033[1;34m▸ {k:<23}\033[0m \033[1;37m{v}\033[0m")

    print("  " + "─" * 56)
    print(f"  \033[90m📊 Total facts stored:\033[0m \033[1;32m{len(facts)}\033[0m")

print("\n" + "═" * 62 + "\n")


══════════════════════════════════════════════════════════════
  👤  USER PROFILE — STORED FACTS
  Redis Hash: profile:user_mustafa
══════════════════════════════════════════════════════════════

  FIELD                     VALUE
  ────────────────────────────────────────────────────────
  ▸ name                    Mustafa
  ▸ role                    Machine Learning Engineer
  ▸ experience              3 years
  ▸ preferred_language      Python
  ────────────────────────────────────────────────────────
  📊 Total facts stored: 4

══════════════════════════════════════════════════════════════



In [17]:
# Now start a NEW session — profile persists even though conversation is fresh
new_session = "mustafa_new_session"
memory.clear_session(new_session)

print("\n" + "═" * 72)
print("\033[1;36m  🆕  NEW SESSION — PROFILE PERSISTENCE TEST\033[0m")
print("═" * 72)

print(f"\n  \033[1;34m▸ Session ID\033[0m        → \033[90m{new_session}\033[0m")
print(f"  \033[1;33m▸ Conversation\033[0m      → \033[1;37mEMPTY\033[0m  \033[90m(fresh session)\033[0m")
print(f"  \033[1;32m▸ Profile\033[0m           → \033[1;32mPERSISTS\033[0m  \033[90m(still in Redis from before)\033[0m")

# ─────────────────────────────────────────────
# QUESTIONS WITH PERSISTENT PROFILE
# ─────────────────────────────────────────────
questions = [
    "What should I focus on to improve as an ML engineer?",
    "Can you recommend a Redis caching pattern that works well with Python?",
]

for i, q in enumerate(questions, 1):
    print("\n" + "─" * 72)
    print(f"\033[1;35m  💬  Question {i}/{len(questions)}\033[0m")
    print("─" * 72)

    reply = chat_with_profile(new_session, "user_alex", q, profile)

    print(f"\n  \033[1;36m[YOU]\033[0m  {q}")
    print(f"\n  \033[1;32m[BOT]\033[0m  \033[90m{reply[:300]}\033[0m")

# ─────────────────────────────────────────────
# CLEANUP
# ─────────────────────────────────────────────
print("\n" + "═" * 72)
print("\033[1;33m  🧹  CLEANUP\033[0m")
print("═" * 72)

profile.clear("user_mustafa")
memory.clear_session(new_session)
memory.clear_session(session_id)
memory.clear_session(session_user1)
memory.clear_session(session_user2)

sessions_cleared = [
    f"profile:user_mustafa",
    f"conversation:{new_session}",
    f"conversation:{session_id}",
    f"conversation:{session_user1}",
    f"conversation:{session_user2}",
]

print(f"\n  \033[1;32m✅ Cleared {len(sessions_cleared)} key(s) from Redis:\033[0m\n")

for key in sessions_cleared:
    print(f"     \033[90m🗑️  {key}\033[0m")

print("\n" + "═" * 72 + "\n")


════════════════════════════════════════════════════════════════════════
  🆕  NEW SESSION — PROFILE PERSISTENCE TEST
════════════════════════════════════════════════════════════════════════

  ▸ Session ID        → mustafa_new_session
  ▸ Conversation      → EMPTY  (fresh session)
  ▸ Profile           → PERSISTS  (still in Redis from before)

────────────────────────────────────────────────────────────────────────
  💬  Question 1/2
────────────────────────────────────────────────────────────────────────

  [YOU]  What should I focus on to improve as an ML engineer?

  [BOT]  Improving as a machine‑learning (ML) engineer is a blend of deepening **foundational knowledge**, sharpening **practical engineering skills**, and staying current with the fast‑moving research landscape. Below is a roadmap that balances theory, code, systems, and career‑growth activities. Feel free 

────────────────────────────────────────────────────────────────────────
  💬  Question 2/2
───────────────────────

In [18]:
# Safer cleanup — only clear what exists
profile.clear("user_mustafa")
memory.clear_session(new_session)

# Conditionally clear others if they exist
for sess_var in ["session_id", "session_user1", "session_user2"]:
    if sess_var in globals():
        memory.clear_session(globals()[sess_var])